# Deep Q-Network (DQN)
"Playing Atari with Deep Reinforcement Learning" (Mnih et al., 2013) and "Human-level control through deep reinforcement learning" (Mnih et al., 2015)

**Idea:** Q-learning with a neural network in place of the table. The network takes a state and outputs one value per action: the total future reward expected if that action is taken now and the best actions afterwards. Two tricks make this stable: learn from a **replay buffer** of past transitions instead of the latest one, and compute targets with a slowly updated **target network**.

### DQN vs tabular Q-learning vs policy gradient
| | Tabular Q-learning | DQN | Policy gradient (REINFORCE / PPO) |
|---|---|---|---|
| Learns | table $Q[s, a]$ | network $Q_\theta(s, \cdot)$ | policy $\pi_\theta(a \mid s)$ |
| States | few, discrete | large / continuous (pixels) | large / continuous |
| Actions | discrete | discrete | discrete or continuous |
| Acts by | $\arg\max_a Q$ + $\epsilon$-greedy | $\arg\max_a Q$ + $\epsilon$-greedy | sampling from $\pi$ |
| Data | online | **off-policy**, replay buffer (reuses old data) | on-policy (fresh rollouts only) |
| Target | bootstrapped | bootstrapped, from a **target network** | Monte-Carlo return / advantage |
| Used for | toy problems | Atari | robotics, LLM post-training (`models.md`: PPO, GRPO) |

Here: CartPole written from scratch (no gym dependency), a replay buffer, an MLP Q-network, and the DQN loop with a Double-DQN switch.

Loop: `state (4,) -> Q-net -> Q values (2,) -> epsilon-greedy action -> env.step -> (s, a, r, s', done) into buffer -> sample batch (B, ...) -> TD target from target net -> Huber loss -> update Q-net`

In [ ]:
import math
import random
from collections import deque

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cpu"   # tiny MLP, one state at a time: CPU is faster than moving every step to the GPU
torch.manual_seed(42)
random.seed(42)

## 1. Environment - CartPole
A pole is hinged on a cart that moves along a track. Each step the agent pushes the cart left or right. Reward is **+1 for every step** the pole stays up. The episode ends when the pole tilts more than 12 degrees or the cart leaves the track (**terminated**), or after 500 steps (**truncated**: time ran out, the state itself was fine).

- state $s \in \mathbb{R}^4$: cart position $x$, cart velocity $\dot x$, pole angle $\theta$, pole angular velocity $\dot\theta$
- actions: 0 = push left, 1 = push right
- best possible return: 500

The physics below is the same as Gym's `CartPole-v1` (Barto, Sutton & Anderson, 1983), integrated with Euler steps of 0.02 s.

In [ ]:
class CartPole:
    gravity, mass_cart, mass_pole, length = 9.8, 1.0, 0.1, 0.5     # length = half the pole length
    force_mag, tau = 10.0, 0.02
    x_limit, theta_limit, max_steps = 2.4, 12 * math.pi / 180, 500

    def __init__(self, seed=0):
        self.rng = random.Random(seed)

    def reset(self):
        self.state = [self.rng.uniform(-0.05, 0.05) for _ in range(4)]
        self.t = 0
        return list(self.state)

    def step(self, action):
        # -> next_state (4,), reward, terminated, truncated
        x, x_dot, theta, theta_dot = self.state
        force = self.force_mag if action == 1 else -self.force_mag
        total_mass = self.mass_cart + self.mass_pole
        pml = self.mass_pole * self.length
        cos, sin = math.cos(theta), math.sin(theta)
        temp = (force + pml * theta_dot ** 2 * sin) / total_mass
        theta_acc = (self.gravity * sin - cos * temp) / (self.length * (4.0 / 3.0 - self.mass_pole * cos ** 2 / total_mass))
        x_acc = temp - pml * theta_acc * cos / total_mass
        x, x_dot = x + self.tau * x_dot, x_dot + self.tau * x_acc
        theta, theta_dot = theta + self.tau * theta_dot, theta_dot + self.tau * theta_acc
        self.state = [x, x_dot, theta, theta_dot]
        self.t += 1
        terminated = abs(x) > self.x_limit or abs(theta) > self.theta_limit
        truncated = not terminated and self.t >= self.max_steps
        return list(self.state), 1.0, terminated, truncated

env = CartPole()
s, ret, done = env.reset(), 0.0, False
while not done:
    s, r, term, trunc = env.step(random.randrange(2))
    ret += r
    done = term or trunc
print("random policy, one episode return:", ret)

## 2. Q-learning
### Formula: Return and action-value
$$G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k}, \qquad Q^\pi(s, a) = \mathbb{E}_\pi\big[G_t \mid s_t = s,\ a_t = a\big]$$
- $r_t$: reward at step $t$; $\gamma = 0.99$: discount, makes far-future reward count less and keeps the sum finite
- $Q^\pi(s, a)$: expected return from taking $a$ in $s$ and following policy $\pi$ afterwards
- with reward 1 per step, $Q$ is at most $1 / (1 - \gamma) = 100$

### Formula: Bellman optimality equation
$$Q^*(s, a) = \mathbb{E}_{s'}\Big[r + \gamma \max_{a'} Q^*(s', a')\Big]$$
- $Q^*$: value under the best policy; the best policy is then simply $\pi^*(s) = \arg\max_a Q^*(s, a)$
- the value of now = reward now + discounted value of the best next move

### Formula: TD target and loss (DQN)
$$y = r + \gamma\,(1 - d)\,\max_{a'} Q_{\theta^-}(s', a'), \qquad \mathcal{L}(\theta) = \text{Huber}\big(Q_\theta(s, a) - y\big)$$
- $(s, a, r, s', d)$: one stored transition; $d = 1$ if the episode **terminated** at $s'$ (no future reward), else 0
- on **truncation** $d = 0$: the pole was still up, so we keep bootstrapping from $s'$
- $\theta^-$: parameters of the **target network**, a frozen copy of $\theta$ refreshed every $C$ steps. $y$ is treated as a constant (no gradient)
- Huber loss: quadratic for small errors, linear for large ones, so a few huge TD errors cannot blow up the update (the 2015 paper's "error clipping")

### Formula: $\epsilon$-greedy exploration
$$a = \begin{cases} \text{random action} & \text{with probability } \epsilon \\ \arg\max_a Q_\theta(s, a) & \text{otherwise} \end{cases}$$
- $\epsilon$ decays linearly from 1.0 to 0.05 over the first 10,000 steps: explore first, exploit later

In [ ]:
class QNet(nn.Module):
    # state (B, 4) -> Q values (B, 2), one per action
    def __init__(self, state_dim=4, num_actions=2, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, num_actions),
        )

    def forward(self, s):
        return self.net(s)

q_net = QNet().to(device)
print(f"params: {sum(p.numel() for p in q_net.parameters())} |", q_net(torch.randn(3, 4)).shape)

## 3. Replay buffer
Consecutive transitions are highly correlated (one frame looks like the next) and their distribution shifts as the policy changes; SGD assumes neither. Storing transitions and sampling **uniformly at random** breaks the correlation and lets every transition be reused many times.

This is only valid because Q-learning is **off-policy**: the target uses $\max_{a'}$, not the action the old policy actually took next.

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.data = deque(maxlen=capacity)                # oldest transitions fall out

    def push(self, s, a, r, s2, terminated):
        self.data.append((s, a, r, s2, float(terminated)))

    def sample(self, batch_size):
        # -> s (B, 4), a (B,), r (B,), s2 (B, 4), d (B,)
        s, a, r, s2, d = zip(*random.sample(self.data, batch_size))
        return (torch.tensor(s, device=device), torch.tensor(a, device=device), torch.tensor(r, device=device),
                torch.tensor(s2, device=device), torch.tensor(d, device=device))

    def __len__(self):
        return len(self.data)

## 4. The update
One gradient step on a sampled batch. `double=True` switches the target to **Double DQN** (van Hasselt et al., 2016).

### Formula: Double DQN target
$$y = r + \gamma\,(1 - d)\, Q_{\theta^-}\big(s',\ \arg\max_{a'} Q_\theta(s', a')\big)$$
- plain DQN uses the target net both to **choose** and to **evaluate** the next action; $\max$ over noisy estimates is biased upward, so Q values are overestimated
- Double DQN chooses with the online net $\theta$ and evaluates with the target net $\theta^-$: one line of code, less bias

In [ ]:
gamma = 0.99

def dqn_loss(q_net, target_net, batch, double=False):
    s, a, r, s2, d = batch
    q = q_net(s).gather(1, a[:, None]).squeeze(1)                       # (B, 2) -> (B,) Q(s, a) of the action taken
    with torch.no_grad():
        if double:
            a2 = q_net(s2).argmax(1, keepdim=True)                      # (B, 1) online net chooses
            q2 = target_net(s2).gather(1, a2).squeeze(1)                # (B,)   target net evaluates
        else:
            q2 = target_net(s2).max(1).values                           # (B,)
        y = r + gamma * (1 - d) * q2                                    # (B,) TD target
    return F.smooth_l1_loss(q, y)                                       # Huber

# random transitions, just to check shapes
buf = ReplayBuffer(100)
for _ in range(64):
    buf.push([random.random() for _ in range(4)], random.randrange(2), 1.0, [random.random() for _ in range(4)], random.random() < 0.1)
print(dqn_loss(q_net, QNet(), buf.sample(32)).item(), dqn_loss(q_net, QNet(), buf.sample(32), double=True).item())

## 5. Training loop
Interleave acting and learning: one environment step, one gradient step.

| hyperparameter | value |
|---|---|
| total env steps | 40,000 |
| buffer capacity | 50,000 |
| learning starts after | 1,000 steps (buffer warm-up with random actions) |
| batch size | 64 |
| target network update | hard copy every 500 steps |
| optimizer | Adam, lr $5 \times 10^{-4}$, grad-norm clip 10 |

In [ ]:
total_steps = 40000
buffer_size = 50000
learning_starts = 1000
batch_size = 64
target_update = 500
lr = 5e-4
eps_start, eps_end, eps_decay = 1.0, 0.05, 10000

def epsilon(step):
    return max(eps_end, eps_start - (eps_start - eps_end) * step / eps_decay)

def act(q_net, s, eps):
    if random.random() < eps:
        return random.randrange(2)
    with torch.no_grad():
        return q_net(torch.tensor([s], device=device)).argmax(1).item()   # (1, 4) -> (1, 2) -> action

def train_dqn(double=False, seed=0):
    torch.manual_seed(seed); random.seed(seed)
    env = CartPole(seed)
    q_net = QNet().to(device)
    target_net = QNet().to(device)
    target_net.load_state_dict(q_net.state_dict())
    optimizer = torch.optim.Adam(q_net.parameters(), lr=lr)
    buffer = ReplayBuffer(buffer_size)

    returns, s, ep_ret = [], env.reset(), 0.0
    for step in range(1, total_steps + 1):
        a = act(q_net, s, epsilon(step))
        s2, r, terminated, truncated = env.step(a)
        buffer.push(s, a, r, s2, terminated)            # store `terminated`, not `done`: truncation still bootstraps
        s, ep_ret = s2, ep_ret + r
        if terminated or truncated:
            returns.append((step, ep_ret))
            s, ep_ret = env.reset(), 0.0

        if step >= learning_starts:
            loss = dqn_loss(q_net, target_net, buffer.sample(batch_size), double)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(q_net.parameters(), 10.0)
            optimizer.step()
        if step % target_update == 0:
            target_net.load_state_dict(q_net.state_dict())           # theta^- <- theta

        if step % 5000 == 0:
            last = [g for _, g in returns[-20:]]
            print(f"step {step:6d} | episodes {len(returns):4d} | eps {epsilon(step):.2f} | mean return (last 20) {sum(last) / max(1, len(last)):.1f}")
    return q_net, returns

q_net, returns = train_dqn(double=False)

## 6. Results
Episode return over training (with a 20-episode moving average), then the greedy policy ($\epsilon = 0$) on fresh episodes.

DQN learning curves are **not monotone**: the policy can reach 500 and then collapse for a while (a small change in Q values flips the argmax in many states, and the new policy generates a different data distribution).

In [ ]:
def plot_returns(returns, label):
    steps, rets = zip(*returns)
    avg = [sum(rets[max(0, i - 19):i + 1]) / len(rets[max(0, i - 19):i + 1]) for i in range(len(rets))]
    plt.plot(steps, rets, alpha=0.25)
    plt.plot(steps, avg, color=plt.gca().lines[-1].get_color(), label=label)

def evaluate(q_net, episodes=10, seed=123):
    env, out = CartPole(seed), []
    for _ in range(episodes):
        s, ret, done = env.reset(), 0.0, False
        while not done:
            s, r, term, trunc = env.step(act(q_net, s, eps=0.0))
            ret += r
            done = term or trunc
        out.append(ret)
    return out

plot_returns(returns, "DQN")
plt.xlabel("env step"); plt.ylabel("episode return"); plt.legend(); plt.show()
print("greedy policy returns:", evaluate(q_net))

## 7. What did it learn?
The learned $\max_a Q(s, a)$ and the greedy action as a function of pole angle and angular velocity, with the cart at rest in the center. Values are highest when the pole is upright and still; the action boundary runs diagonally (push toward the side the pole is falling to).

In [ ]:
theta = torch.linspace(-0.21, 0.21, 80)
theta_dot = torch.linspace(-2.0, 2.0, 80)
T, TD = torch.meshgrid(theta, theta_dot, indexing="xy")
states = torch.stack([torch.zeros_like(T), torch.zeros_like(T), T, TD], dim=-1).reshape(-1, 4)   # (6400, 4)
with torch.no_grad():
    q = q_net(states.to(device)).cpu()                                                           # (6400, 2)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
im = ax[0].imshow(q.max(1).values.reshape(80, 80), origin="lower", extent=[-0.21, 0.21, -2, 2], aspect="auto")
ax[0].set_title("max_a Q(s, a)"); plt.colorbar(im, ax=ax[0])
ax[1].imshow(q.argmax(1).reshape(80, 80), origin="lower", extent=[-0.21, 0.21, -2, 2], aspect="auto", cmap="coolwarm")
ax[1].set_title("greedy action (blue = left, red = right)")
for a in ax: a.set_xlabel("pole angle (rad)"); a.set_ylabel("pole angular velocity")
plt.show()

## 8. Double DQN
Same loop with the Double DQN target. To compare the two fairly, run several seeds each (RL results from a single seed mean little).

In [ ]:
q_double, returns_double = train_dqn(double=True)
plot_returns(returns, "DQN")
plot_returns(returns_double, "Double DQN")
plt.xlabel("env step"); plt.ylabel("episode return"); plt.legend(); plt.show()
print("Double DQN greedy returns:", evaluate(q_double))

## Notes
- The "deadly triad": **function approximation + bootstrapping + off-policy data** together can diverge. Replay and the target network are the two patches that make it work in practice; neither is a guarantee.
- Without the target network, each update moves the target it is chasing (both depend on $\theta$). Freezing $\theta^-$ for $C$ steps turns learning into a sequence of ordinary regression problems. A common alternative is a soft update $\theta^- \leftarrow \tau\theta + (1 - \tau)\theta^-$ every step.
- The original DQN used a CNN over the last 4 Atari frames (84x84 grayscale), a 1M-transition buffer, 50M frames, and reward clipping to $[-1, 1]$. The loop is the one above.
- Improvements, combined in **Rainbow** (2017): Double DQN, prioritized replay (sample transitions with large TD error more often), dueling heads ($Q = V + A$), n-step returns, distributional RL (C51), noisy nets.
- DQN only handles discrete actions (it needs $\max_a$). Continuous control uses actor-critic methods (DDPG, TD3, SAC), which learn a policy network to do the argmax.
- LLM post-training uses the other family: **policy-gradient** methods (PPO, GRPO) on a reward from a verifier or reward model, where the "action" is the next token and the policy is the language model itself (`models.md`).